# Common Functions — Reusable Utilities
## SalesFlow Data Lakehouse | Phase 3: Utility Library

This notebook contains reusable transformation and validation functions  
shared across Bronze, Silver, and Gold layers.

**How to import in other notebooks:**
```python
%run /04_Utils/common_functions
```

| Function | Purpose | Used In |
|---|---|---|
| `clean_string_column()` | Trim and title-case string columns | Silver |
| `standardize_phone()` | Normalize phone numbers | Silver — Customers |
| `add_quality_flag()` | Flag records as VALID/INVALID | Silver — All tables |
| `add_surrogate_key()` | Generate surrogate keys | Gold — Dimensions |

In [0]:
# Imports
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, trim, initcap, when, isnull, lit,
    regexp_replace, monotonically_increasing_id,
    md5, concat_ws
)

## 3.1 String Cleaning — `clean_string_column()`
Trims whitespace and applies title case to a string column. Nulls are preserved as null.

In [0]:
def clean_string_column(df: DataFrame, column_name: str) -> DataFrame:
    """
    Cleans a string column by trimming whitespace and applying title case.
    Null values are kept as null.

    Args:
        df (DataFrame): Input DataFrame.
        column_name (str): Name of the column to clean.

    Returns:
        DataFrame: DataFrame with the cleaned column.

    Example:
        df = clean_string_column(df, "CustomerName")
    """
    return df.withColumn(
        column_name,
        when(
            isnull(col(column_name)), None  # preserve nulls
        ).otherwise(
            initcap(trim(col(column_name)))  # trim whitespace + title case
        )
    )

## 3.2 Phone Standardization — `standardize_phone()`
Removes special characters from phone numbers, keeping only digits and `+`.  
Returns null if the result has fewer than 7 digits (considered invalid).

In [0]:
def standardize_phone(df: DataFrame, column_name: str) -> DataFrame:
    """
    Standardizes a phone number column by removing special characters.
    Keeps only digits and '+'. Returns null if the result is invalid (< 7 digits).

    Args:
        df (DataFrame): Input DataFrame.
        column_name (str): Name of the phone column.

    Returns:
        DataFrame: DataFrame with the standardized phone column.

    Example:
        df = standardize_phone(df, "Phone")
    """
    cleaned = regexp_replace(col(column_name), r"[^\d+]", "")  # keep digits and +

    return df.withColumn(
        column_name,
        when(isnull(col(column_name)), None)           # preserve nulls
        .when(
            regexp_replace(cleaned, r"[^\d]", "").rlike(r"^\d{7,}$"),
            cleaned                                    # valid: 7+ digits
        )
        .otherwise(None)                               # invalid: return null
    )

## 3.3 Quality Flag — `add_quality_flag()`
Adds a `data_quality_status` column to the DataFrame.  
A record is marked `INVALID` if any of the required columns is null.

In [0]:
def add_quality_flag(df: DataFrame, required_columns: list) -> DataFrame:
    """
    Adds a data_quality_status column based on nulls in required columns.
    Marks a record as INVALID if any required column is null, otherwise VALID.

    Args:
        df (DataFrame): Input DataFrame.
        required_columns (list): List of column names that must not be null.

    Returns:
        DataFrame: DataFrame with the new data_quality_status column.

    Example:
        df = add_quality_flag(df, ["CustomerID", "CustomerName", "Country"])
    """
    # Build a condition: any required column is null → INVALID
    invalid_condition = isnull(col(required_columns[0]))
    for c in required_columns[1:]:
        invalid_condition = invalid_condition | isnull(col(c))

    return df.withColumn(
        "data_quality_status",
        when(invalid_condition, lit("INVALID")).otherwise(lit("VALID"))
    )

## 3.4 Surrogate Key — `add_surrogate_key()`
Adds a surrogate key column named `{table_name}_key` using a monotonically increasing ID.

In [0]:
def add_surrogate_key(df: DataFrame, table_name: str, key_columns: list) -> DataFrame:
    """
    Adds a surrogate key column named {table_name}_key using an MD5 hash
    of the specified key columns. The key is deterministic — same input
    always produces the same hash, making it safe for incremental loads.

    Args:
        df (DataFrame): Input DataFrame.
        table_name (str): Name of the table, used to name the key column.
        key_columns (list): Columns used to generate the hash.

    Returns:
        DataFrame: DataFrame with the new surrogate key column as the first column.

    Example:
        df = add_surrogate_key(df, "customers", ["CustomerID"])
        # Adds column: customers_key (MD5 hash of CustomerID)
    """
    key_col = f"{table_name}_key"

    return df.withColumn(
        key_col,
        md5(concat_ws("||", *[col(c).cast("string") for c in key_columns]))
    ).select(key_col, *[c for c in df.columns])  # move key to first position

## Tests
Validates all four functions using a small inline DataFrame.

In [0]:
from pyspark.sql import Row

# --- Test data ---
test_data = [
    Row(name="  john silva ", phone="(11) 98765-4321", customer_id=1, country="Brazil"),
    Row(name="JANE DOE",      phone="invalid",         customer_id=2, country=None),
    Row(name=None,            phone=None,              customer_id=None, country="USA"),
]
test_df = spark.createDataFrame(test_data)

print("=== Original ===")
display(test_df)

# 3.1 clean_string_column
test_df = clean_string_column(test_df, "name")
test_df = clean_string_column(test_df, "country")
print("\n=== After clean_string_column ===")
display(test_df)

# 3.2 standardize_phone
test_df = standardize_phone(test_df, "phone")
print("\n=== After standardize_phone ===")
display(test_df)

# 3.3 add_quality_flag
test_df = add_quality_flag(test_df, ["customer_id", "name", "country"])
print("\n=== After add_quality_flag ===")
display(test_df)

# 3.4 add_surrogate_key
test_df = add_surrogate_key(test_df, "customers", ["customer_id"])
print("\n=== After add_surrogate_key ===")
display(test_df)